# CS2309 — SwiftEdit WebUI T4 (`fp16_disk_xformers`)

Demo Gradio trên **Google Colab T4**.

| Thành phần | Chi tiết |
|------------|----------|
| Script | [`scripts/app_gradio_t4_xformers.py`](../scripts/app_gradio_t4_xformers.py) |
| Config | **`fp16_disk_xformers`** = FP16 disk + xFormers MEA + EditCache |
| Weights | Drive trước; thiếu → tải (+ lưu Drive) |
| Share | Mặc định Gradio `*.gradio.live`; tùy chọn ngrok |
| UI | **Giống Mac** (`app_gradio.py`): ROI mask + 3 candidates + Regen/Undo |

### Pipeline (bắt buộc theo thứ tự)

1. Mount Drive
2. Kiểm tra weights (log CÓ/THIẾU) → thiếu thì **tải** (log tiến độ)
3. **Load model lên GPU** (VRAM phải tăng) — fail thì **không** mở app
4. Launch Gradio + share URL

### Cách chạy (Colab)

1. Runtime → **T4 GPU**
2. Cell **①** clone + GPU
3. Cell **②** pip
4. Cell **③** mount + kiểm tra weights
5. Cell **④** launch (tải nếu thiếu → load → share)

Repo private: Colab Secrets → `GITHUB_TOKEN`.  
ngrok (tuỳ chọn): Secrets → `NGROK_AUTHTOKEN`, đặt `SHARE_MODE = "ngrok"` hoặc `"both"`.

> Upload sẵn `swiftedit_weights_fp16` lên Drive để bỏ tải Qualcomm ~10GB.


### ① Clone repo + kiểm tra GPU T4

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_COLAB_GPU_ERR = (
    "Colab chưa có GPU CUDA (cần T4).\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Extension: New Colab Server → GPU → T4, rồi Restart kernel"
)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK:", ", ".join(names))


REPO_SLUG = "NguyenKz/CS2309.CH201"
USE_PRIVATE_REPO = True


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        if not token:
            raise ValueError("Thiếu GITHUB_TOKEN")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        print(f"Không lấy GITHUB_TOKEN ({e}) — fallback repo public.")
        return public_url


COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if not IN_COLAB:
    raise RuntimeError(
        "Notebook này dành cho Google Colab T4.\n"
        "Local Mac: dùng notebooks/CS2309_SwiftEdit_webui.ipynb + scripts/app_gradio.py"
    )

_check_colab_gpu()
REPO_URL = _colab_repo_url()

if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
    print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
        check=True,
    )
else:
    print("Repo đã có — git pull ...")
    subprocess.run(
        ["git", "-C", str(COLAB_REPO_DIR), "pull", "--ff-only"],
        check=False,
    )
    print("Repo:", COLAB_REPO_DIR)

PROJECT_ROOT = COLAB_REPO_DIR
os.environ.setdefault("HF_HOME", "/content/huggingface")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

APP_SCRIPT = PROJECT_ROOT / "scripts" / "app_gradio_t4_xformers.py"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("app_gradio_t4_xformers.py:", APP_SCRIPT.is_file())
if not APP_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Thiếu {APP_SCRIPT} — pull/push nhánh có scripts/app_gradio_t4_xformers.py"
    )


### ② Setup pip (gradio + xformers)

Weights **không** tải ở cell này. Cell **③** mount + kiểm tra; cell **④** mới tải (nếu thiếu) → load → share.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "gradio>=5,<6",
        "huggingface-hub<1.0",
        "xformers",
    ],
    check=True,
)

req = PROJECT_ROOT / "SwiftEdit" / "requirements.txt"
if req.is_file():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True,
    )

import gradio as gr
import torch
import xformers

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("xformers:", getattr(xformers, "__version__", "?"))
print("gradio:", gr.__version__)
print("Setup OK — chạy cell ③ (mount + kiểm tra weights).")

### ③ Mount Drive + kiểm tra weights

Mount Drive (auth phải ở kernel), rồi in rõ Drive/local **CÓ** hay **THIẾU** fp16/fp32.

Chưa launch app ở cell này.


In [ ]:
from pathlib import Path

# --- Cấu hình (dùng lại ở cell ④) ---
USE_DRIVE = True
DRIVE_FP16 = Path("/content/drive/MyDrive/CS2309/swiftedit_weights_fp16")
DRIVE_FP32 = Path("/content/drive/MyDrive/CS2309/swiftedit_weights")
LOCAL_FP16 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights_fp16"
LOCAL_FP32 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"

ALLOW_DOWNLOAD = True   # thiếu Drive/local → tải Qualcomm
SAVE_TO_DRIVE = True    # sau tải/convert → copy lên Drive
# gradio | ngrok | both | none
SHARE_MODE = "gradio"
PORT_START = 7860


def _tree_ok(path: Path) -> bool:
    return (
        (path / "sbv2_0.5").is_dir()
        and (path / "ip_adapter_ckpt-90k" / "ip_adapter.bin").is_file()
        and (path / "inverse_ckpt-120k").exists()
    )


def _mount_drive() -> None:
    """Mount phải chạy trong kernel — subprocess không auth được."""
    if not USE_DRIVE:
        print("USE_DRIVE=False — bỏ qua mount.")
        return
    from google.colab import drive

    marker = Path("/content/drive/MyDrive")
    if marker.is_dir():
        print("Drive đã mount:", marker)
    else:
        print("Mount Google Drive (click Connect nếu Colab hỏi)…")
        drive.mount("/content/drive", force_remount=False)
    if not marker.is_dir():
        raise RuntimeError("Mount Drive thất bại — không thấy /content/drive/MyDrive")


def _inspect_weights() -> None:
    rows = [
        ("Drive fp16", DRIVE_FP16),
        ("Drive fp32", DRIVE_FP32),
        ("Local fp16", LOCAL_FP16),
        ("Local fp32", LOCAL_FP32),
    ]
    print("\nKiểm tra weights:")
    for label, path in rows:
        ok = _tree_ok(path)
        mark = "CÓ" if ok else "THIẾU"
        state = "dir" if path.is_dir() else ("có path" if path.exists() else "không tồn tại")
        print(f"  [{mark:5}] {label}: {path} ({state})")
    if _tree_ok(DRIVE_FP16) or _tree_ok(LOCAL_FP16):
        print("→ fp16 sẵn sàng. Cell ④ sẽ load model (VRAM tăng) rồi mở app.")
    elif _tree_ok(DRIVE_FP32) or _tree_ok(LOCAL_FP32):
        print("→ chỉ có fp32. Cell ④ sẽ convert → fp16 (lâu) rồi load.")
    else:
        print(
            "→ chưa có weights. Cell ④ sẽ TẢI Qualcomm (~10GB).\n"
            "  Trong lúc tải VRAM vẫn ~0 — đó là bình thường."
        )


_mount_drive()
_inspect_weights()

print("\ngit pull (đồng bộ script)…")
subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=False)

app_txt = APP_SCRIPT.read_text(encoding="utf-8", errors="ignore")
if "share-mode" not in app_txt and "BƯỚC" not in app_txt:
    raise FileNotFoundError(
        f"{APP_SCRIPT} trên Colab quá cũ (thiếu pipeline mới).\n"
        "Push commit mới lên GitHub → Restart → chạy lại cell ①."
    )
print("OK — chạy cell ④ Launch.")


### ④ Launch WebUI (tải nếu thiếu → load GPU → share)

Script in rõ **BƯỚC 1–4**:
1. CUDA
2. Kiểm tra/tải weights (log CÓ/THIẾU; tải thì có tiến độ)
3. Load model — **VRAM phải tăng**; fail thì không mở app
4. Gradio + share (`SHARE_MODE`)

Đợi dòng `Gradio Public URL` / `*.gradio.live`. Dừng: **Interrupt kernel** (■).

In [ ]:
import socket


def _free_port(start: int = 7860, count: int = 20) -> int:
    for port in range(start, start + count):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                s.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    return start


def _gpu_used_ratio() -> float | None:
    r = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        return None
    used, total = (float(x.strip()) for x in r.stdout.strip().split(","))
    print(f"GPU VRAM trước launch: {used:.0f} / {total:.0f} MiB")
    return used / total if total else None


def _run_live(cmd: list[str], *, cwd: Path, env: dict) -> int:
    """Chạy subprocess, in từng dòng ngay (Colab không nuốt log)."""
    print("Lệnh:", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


ratio = _gpu_used_ratio()
if ratio is not None and ratio > 0.85:
    raise RuntimeError(
        "GPU gần đầy — Restart kernel rồi chạy lại ①②③④.\n"
        "Tránh nạp model hai lần trong cùng kernel."
    )

# Nhắc lại trạng thái weights trước khi spawn
_inspect_weights()

PORT = _free_port(PORT_START)
cmd = [
    sys.executable,
    "-u",
    str(APP_SCRIPT),
    "--drive-fp16",
    str(DRIVE_FP16),
    "--drive-fp32",
    str(DRIVE_FP32),
    "--local-fp16",
    str(LOCAL_FP16),
    "--local-fp32",
    str(LOCAL_FP32),
    "--port",
    str(PORT),
    "--share-mode",
    SHARE_MODE,
]
if not ALLOW_DOWNLOAD:
    cmd.append("--no-allow-download")
if not SAVE_TO_DRIVE:
    cmd.append("--no-save-to-drive")

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"

print("\n" + "=" * 60)
print("Config: fp16_disk_xformers + EditCache")
print("SHARE_MODE:", SHARE_MODE, "| SAVE_TO_DRIVE:", SAVE_TO_DRIVE, "| ALLOW_DOWNLOAD:", ALLOW_DOWNLOAD)
print("Thứ tự: weights → load GPU (VRAM tăng) → mới share URL")
print("Dừng: Interrupt kernel (■)")
print("=" * 60 + "\n")

code = _run_live(cmd, cwd=PROJECT_ROOT, env=run_env)
if code != 0:
    raise RuntimeError(
        f"app_gradio_t4_xformers.py thoát code {code}.\n"
        "Kéo lên xem log. Thiếu script mới → git pull + restart cell ①."
    )